In [22]:
import ollama 
import os
from tqdm import tqdm
import json
import signal
import argparse
import wandb
import pandas as pd
import matplotlib.pyplot as plt
import copy
import numpy as np
import re

import sys
from collections import defaultdict

In [23]:
sys.argv = [
    'notebook',  
    '--modelname', 'llava-llama3:8b', #'llama3.2-vision:90b',
    '--data', 'gully',
    '--data_path','/mnt/jacket/WACV-2025-Workshop-ViGIR/results/baseline/reason_first_results_llava-llama3:8b.json',
    '--subset', 'train',
    '--results_dir', '/root/home/WACV-2025-Workshop-ViGIR/results/baseline',
    '--timeout', '20',
    '--model_unloading'
]

In [24]:
parser = argparse.ArgumentParser(description="A script to evaluate V-LLMs on different image classification datasets")

parser.add_argument("--modelname", type=str, required=True, help="The name of the V-LLM model")
parser.add_argument("--data", type=str, required=True, help="Dataset name")
parser.add_argument("--data_path", type=str, required=True, help="Path to the image data dir")
parser.add_argument("--subset", type=str, required=True, help="train, test or validation set")
parser.add_argument("--results_dir", type=str, required=True, help="Folder name to save results")
parser.add_argument("--timeout", type=int, default=40, help="time out duration to skip one sample")
parser.add_argument("--model_unloading", action="store_true", help="Enables unloading mode. Every 100 samples it unloades the model from the GPU to avoid carshing.")

args = parser.parse_args()

In [25]:
# Load test set:
file_path = os.path.join(args.data_path)
with open(file_path, 'r') as file:
    data = json.load(file)

print('Number of Annotated GT Images: ', len(data.keys()))
data_keys_test = list(data.keys())
print('Number of Annotated GT Images (List): ', len(data_keys_test))

Number of Annotated GT Images:  311
Number of Annotated GT Images (List):  311


In [26]:
data

{'100': ['/root/home/data/neg_collage_100.jpg',
  'Yes, there are ephemeral gully appearances when looking at all of them together. The reason is that each image captures a different moment in time, and these moments can show changes in the landscape over the 10-year period. For example, some images may show more trees or less trees due to seasonal changes, weather patterns, or human activity. By comparing the images side by side, one can observe these differences and identify any ephemeral gully appearances that were present at different times but not consistently across all images.'],
 '1004': ['/root/home/data/neg_collage_1004.jpg',
  'Yes, there are ephemeral gully appearances when looking at all of them together. The reason is that each image captures a different moment in time, and these moments can show changes in the landscape over the 10-year period. These changes could be due to factors such as weather patterns, soil erosion, or human activity. By comparing the images side by

In [19]:
data_reformated = {}

for key, item in data.items():
    """
    q_and_a = []
    
    for i in range(len(item)):
        tmp = item[i][1:]   
        #print(tmp)
        #sys.exit()
        q_and_a.append(tmp)
        print(item[1])
        sys.exit()
    """

    data_reformated[key] = item[1]
    

In [34]:
data_reformated

{'100': 'Yes, there are ephemeral gully appearances when looking at all of them together. The reason is that each image captures a different moment in time, and these moments can show changes in the landscape over the 10-year period. For example, some images may show more trees or less trees due to seasonal changes, weather patterns, or human activity. By comparing the images side by side, one can observe these differences and identify any ephemeral gully appearances that were present at different times but not consistently across all images.',
 '1004': 'Yes, there are ephemeral gully appearances when looking at all of them together. The reason is that each image captures a different moment in time, and these moments can show changes in the landscape over the 10-year period. These changes could be due to factors such as weather patterns, soil erosion, or human activity. By comparing the images side by side, one can observe how the area has evolved over time, revealing any temporary cha

In [36]:
#data_prompt = {}
#for key, item in data_reformated.items():
#    print(item)
#    sys.exit()
#    prompt = "\n".join([f"Q: {qa[0]}\nA: {qa[1]}" for qa in item])
#    data_prompt[key] = [prompt]

In [39]:
data_prompt = copy.deepcopy(data)

In [40]:
model_name = args.modelname
ollama.pull(model_name)

timeout_duration = args.timeout

options= {  # new
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048, # must be set, otherwise slightly random output
        }

model_labels = {}
count = 0

In [96]:
count = 0
saving_response=copy.deepcopy(data_prompt)

for key, info in tqdm(data_prompt.items()):

    #print(key)
    #print(info[0])
    #sys.exit()
    #question = "Based on the following questions and their answers, determine if there is evidence of an ephemeral gully in the observed area. Carefully analyze all the questions and the responses to assess.\n\n"
    #question += info[0]
    #question += "\n\nAfter considering these responses, provide a clear conclusion: Is there evidence of an ephemeral gully? Answer with yes or no only."

    question = "Based on the following questions and their answers, determine if there is evidence of an ephemeral gully in the observed area. Carefully analyze all the questions and the responses to assess.\n\n"
    question += info[0]
    question += "\n\nAfter considering these responses, provide a clear conclusion: Is there evidence of an ephemeral gully? Answer with yes or no only."

    
    #print(question)
    #sys.exit()
        
    count+=1
    #for question in questions:
    response = ollama.generate(model=model_name, 
                               prompt=question, 
                               #images=info, 
                               options=options)
    saving_response[key].append(response['response'])
    print(response['response'])

    #sys.exit()
        

  0%|▎                                                                                                 | 1/311 [02:57<15:18:36, 177.79s/it]

Yes. 

Although most questions received negative answers, two key indicators suggest the presence of an ephemeral gully:

1. A specific path lacks vegetation, suggesting an evolving or emerging channel (Q7).
2. There is a varying exposure of lighter or darker colored soil (Q11).

These signs are indicative of water flow and potential erosion, which are characteristic features of ephemeral gullies.


  1%|▋                                                                                                   | 2/311 [02:58<6:20:11, 73.82s/it]

No.


  1%|▉                                                                                                   | 3/311 [03:02<3:34:40, 41.82s/it]

...

No.


100%|███████████████████████████████████████████████████████████████████████████████████████████████████▋| 310/311 [20:40<00:01,  1.39s/it]

No.
No.


100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 311/311 [20:41<00:00,  3.99s/it]

No.


In [97]:
def extract_first_yes_no(text):
    # Use regular expressions to find "Yes" or "No" (case-insensitive)
    match = re.search(r'\b(Yes|No)\b', text, re.IGNORECASE)
    if match:
        # Return the first match in its original case
        return match.group(0)
    return None

In [98]:
saving_response_reformat = {}

for key, item in saving_response.items():
    #print(key)
    #print(item[-1])
    text = extract_first_yes_no(item[-1])

    print(f"Image: {key} - Answer: {text}")
    
    if text == 'Yes':
        saving_response_reformat[key] = ["4"]

    if text == 'No':
        saving_response_reformat[key] = ["0"]
        

Image: 415 - Answer: Yes
Image: 1020 - Answer: No
Image: 105 - Answer: Yes
Image: 439 - Answer: Yes
Image: 914 - Answer: Yes
Image: 1099 - Answer: Yes
Image: 1065 - Answer: No
Image: 373 - Answer: Yes
Image: 166 - Answer: Yes
Image: 396 - Answer: Yes
Image: 837 - Answer: No
Image: 685 - Answer: Yes
Image: 226 - Answer: Yes
Image: 956 - Answer: No
Image: 70 - Answer: Yes
Image: 604 - Answer: Yes
Image: 1067 - Answer: Yes
Image: 118 - Answer: Yes
Image: 774 - Answer: No
Image: 521 - Answer: No
Image: 910 - Answer: No
Image: 975 - Answer: Yes
Image: 352 - Answer: Yes
Image: 761 - Answer: No
Image: 1007 - Answer: No
Image: 428 - Answer: Yes
Image: 1038 - Answer: No
Image: 50 - Answer: Yes
Image: 838 - Answer: No
Image: 126 - Answer: No
Image: 1078 - Answer: Yes
Image: 944 - Answer: No
Image: 532 - Answer: No
Image: 1041 - Answer: No
Image: 540 - Answer: No
Image: 128 - Answer: No
Image: 722 - Answer: Yes
Image: 478 - Answer: No
Image: 639 - Answer: Yes
Image: 668 - Answer: Yes
Image: 343 -

In [99]:
saving_response_reformat

{'415': ['4'],
 '1020': ['0'],
 '105': ['4'],
 '439': ['4'],
 '914': ['4'],
 '1099': ['4'],
 '1065': ['0'],
 '373': ['4'],
 '166': ['4'],
 '396': ['4'],
 '837': ['0'],
 '685': ['4'],
 '226': ['4'],
 '956': ['0'],
 '70': ['4'],
 '604': ['4'],
 '1067': ['4'],
 '118': ['4'],
 '774': ['0'],
 '521': ['0'],
 '910': ['0'],
 '975': ['4'],
 '352': ['4'],
 '761': ['0'],
 '1007': ['0'],
 '428': ['4'],
 '1038': ['0'],
 '50': ['4'],
 '838': ['0'],
 '126': ['0'],
 '1078': ['4'],
 '944': ['0'],
 '532': ['0'],
 '1041': ['0'],
 '540': ['0'],
 '128': ['0'],
 '722': ['4'],
 '478': ['0'],
 '639': ['4'],
 '668': ['4'],
 '343': ['4'],
 '1021': ['0'],
 '552': ['0'],
 '848': ['0'],
 '74': ['0'],
 '520': ['4'],
 '1032': ['0'],
 '615': ['4'],
 '536': ['0'],
 '1117': ['0'],
 '646': ['4'],
 '390': ['0'],
 '923': ['4'],
 '194': ['4'],
 '216': ['0'],
 '99': ['0'],
 '372': ['4'],
 '857': ['0'],
 '335': ['4'],
 '505': ['0'],
 '972': ['0'],
 '727': ['0'],
 '1064': ['0'],
 '740': ['4'],
 '182': ['4'],
 '316': ['0'],
 '

In [100]:
# Save the dictionary as a JSON file
filename =  'results_'+args.modelname+'.json'
with open(os.path.join(args.results_dir, filename), 'w') as json_file:
    json.dump(saving_response_reformat, json_file, indent=4)

print(f"JSON file {filename} has been created.")

JSON file results_llama3.2-vision:90b.json has been created.
